# 查询所有 submission + 参赛者 + 队伍信息

通过 `Judge()` 复用其 `alphathon_api`（已带好鉴权 cookie）查询：

- **submission**：`api.query_submissions(competition_id=...)`，每条含 `id` / `user_id` / `public_score` 等
- **参赛者**：`/users?competition_id=...`，按 `user_id` 关联（姓名在 `data` 里）
- **队伍**：`/teams?competition_id=...`，按 `user_id` 命中 `creator` / `members` / `pending_users`

三者通过 **平台 user_id** 关联：`submission.user_id == user.user_id`，并据此反查所在队伍。

In [ ]:
import json
import pandas as pd

from public import Judge

judge = Judge()
api = judge.alphathon_api
competition_id = judge.competition_id
print("competition_id:", competition_id)

In [ ]:
def query_all(path: str, competition_id, page_size: int = 5000, max_pages: int = 10000) -> list[dict]:
    """分页拉取某个列表接口的全部数据（/users、/teams 等）。

    复用 api._request（已带鉴权 cookie）。接口返回结构为 {data: {items: [...], total, ...}}。
    """
    results: list[dict] = []
    page = 1
    while page <= max_pages:
        params = {"competition_id": str(competition_id), "page": page, "size": page_size}
        data = api._request("GET", path, params=params).json().get("data") or {}
        items = data.get("items") or []
        if not items:
            break
        results.extend(items)
        if len(items) < page_size:
            break
        page += 1
    return results


# 1) 所有提交
submissions = api.query_submissions(competition_id=competition_id)
# 2) 所有参赛者（报名记录）
users = query_all("/users", competition_id)
# 3) 所有队伍
teams = query_all("/teams", competition_id)

print(f"submissions={len(submissions)}  users={len(users)}  teams={len(teams)}")

In [ ]:
# 参赛者：platform user_id -> 报名记录（姓名等在 data 里）
user_by_uid = {str(u["user_id"]): u for u in users}

# 队伍：platform user_id -> (队伍, 角色)。一个用户在一个比赛中最多属于一个队伍。
# creator=队长, members=正式队员, pending_users=待审批
team_by_uid: dict[str, tuple[dict, str]] = {}
for t in teams:
    creator = str(t.get("creator")) if t.get("creator") else None
    if creator:
        team_by_uid[creator] = (t, "creator")
    for uid in (t.get("members") or []):
        team_by_uid[str(uid)] = (t, "member")
    for uid in (t.get("pending_users") or []):
        # 已在队伍中的不被待审批覆盖
        team_by_uid.setdefault(str(uid), (t, "pending"))

In [ ]:
rows = []
for s in submissions:
    uid = str(s.get("user_id"))
    user = user_by_uid.get(uid) or {}
    user_data = user.get("data") or {}
    team, role = team_by_uid.get(uid, (None, None))
    team = team or {}

    rows.append({
        # ---- submission ----
        "submission_id": s.get("id"),
        "created_at": s.get("created_at"),
        "public_score": s.get("public_score"),
        "private_score": s.get("private_score"),
        "selected_for_private": s.get("selected_for_private"),
        # ---- 参赛者 ----
        "user_id": uid,
        "user_name": user_data.get("name"),
        "user_status": user.get("status"),
        # ---- 队伍 ----
        "team_id": team.get("id"),
        "team_name": team.get("name"),
        "team_role": role,
    })

df = pd.DataFrame(rows)
print(f"共 {len(df)} 条提交")
df.head(20)

## 备注

- `user_name` / `user_status` 来自参赛报名记录（`/users`），姓名等字段在报名 `data` 里。
- `team_role` 取值：`creator`（队长）/ `member`（正式队员）/ `pending`（待审批）；为 `None` 表示该用户未加入任何队伍（个人提交）。
- 若某 `user_id` 在 users 中查不到，多半是该提交对应的报名记录被删除，`user_name` 会是 `None`。